# Week 3 — Cleaning your data
## group2a · Clinic visits

**Your question**
> Which departments have the longest patient wait times, and how does that vary by visit type and over time?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [ ]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [ ]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group2a"
SCHEMA_RAW   = "raw_clinic"
SCHEMA_MINE  = "group2a"

password = getpass("Password for group2a: ")

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.visits", engine)


You should see **25,100**. If not, stop — something is
wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [ ]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.visits", engine)
patients = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.patients", engine)
doctors = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.doctors", engine)
departments = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.departments", engine)

print('visits   ', df.shape)
print('patients'.ljust(12), patients.shape)
print('doctors'.ljust(12), doctors.shape)
print('departments'.ljust(12), departments.shape)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [ ]:
df.head(10)


In [ ]:
df.info()


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [ ]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['visit_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['visit_type'].nunique(),
}
before


---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [ ]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))


**TODO — write down the number removed. Does it match your
week 2 figure?**

If it does not, you are looking at something different from what you counted.
Work out which before continuing.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `visit_type`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [ ]:
# what it looks like now
df['visit_type'].value_counts(dropna=False)


In [ ]:
df['visit_type'] = df['visit_type'].str.strip().str.title()

df['visit_type'].value_counts(dropna=False)


**TODO — how many categories now, and how many before?**

Now do the same for the other text columns. Week 2 should have told you which
ones are affected — it is not only this one.


In [ ]:
# TODO: check and clean the text columns in your dimension tables
patients['sex'] = patients['sex'].str.strip().str.title()
patients['region'] = patients['region'].str.strip().str.title()

# check your work
patients['sex'].value_counts(dropna=False).head(15)


---
## 6. Parse the dates

`visit_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.


In [ ]:
# what formats are present
df['visit_date'].str.len().value_counts()


In [ ]:
df['visit_date'] = pd.to_datetime(
    df['visit_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'      # anything unparseable becomes NaT rather than crashing
)

print("could not parse:", df['visit_date'].isna().sum())
print("range:", df['visit_date'].min(), "to", df['visit_date'].max())


**TODO — does that date range make sense now?**

Compare it with what the text version gave you in week 2. This is where the
text-sorting problem finally goes away.

If any rows failed to parse, decide what to do with them and say why.


---
## 7. Deal with impossible values

Week 2 found negative wait times. Look at them before deciding.


In [ ]:
bad = df[df['wait_minutes'] < 0]
print("rows affected:", len(bad))
bad.head(10)


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong
one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the
   absolute value may be justified. Only if you can argue it.


In [ ]:
# TODO: implement your decision. One of these, or your own.

# option 1 — drop
# df = df[~df.index.isin(bad.index)]

# option 2 — set to NULL
# df.loc[bad.index, 'COLUMN'] = np.nan

print("rows now:", len(df))


---
## 8. Deal with orphan keys

Some `patient_id` values in your fact table point at
`patients` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [ ]:
valid = set(patients['patient_id'])
orphans = df[~df['patient_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['visit_id', 'patient_id']].head(10)


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and
   your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.


In [ ]:
# TODO: implement your decision

# option 1 — drop
# df = df[df['patient_id'].isin(valid)]

# option 2 — add an Unknown row to the dimension, then repoint orphans at it
# unknown = pd.DataFrame([{'patient_id': -1}])
# patients = pd.concat([patients, unknown], ignore_index=True)
# df.loc[~df['patient_id'].isin(valid), 'patient_id'] = -1

print("rows now:", len(df))


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [ ]:
df.isna().sum().sort_values(ascending=False)


**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `visit_type` | | | |
| | | | |

Options: leave as NULL (honest, Power BI shows blanks), fill with a label like
`'Unknown'` (good for text you will group by), or drop the row (only if the row
is useless without it).


In [ ]:
# TODO: implement your decisions

# example — label missing text so it groups properly in Power BI
# df['visit_type'] = df['visit_type'].fillna('Unknown')

df.isna().sum().sort_values(ascending=False).head()


---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [ ]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['visit_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['visit_type'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates,
how many were impossible values, and how many were orphans. If the numbers do
not add up, something happened that you did not intend.


---
## 11. Write the clean tables back

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds the
table each time you run the notebook — use `'append'` by mistake and running
twice silently doubles your data.


In [ ]:
df.to_sql('visits_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
patients.to_sql('patients_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
doctors.to_sql('doctors_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
departments.to_sql('departments_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)

print("written")


### Confirm they landed


In [ ]:
pd.read_sql(f"""
    SELECT table_name,
           (xpath('/row/c/text()',
            query_to_xml(format('SELECT count(*) AS c FROM %I.%I',
            table_schema, table_name), false, true, '')))[1]::text::int AS rows
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA_MINE}'
    ORDER BY table_name
""", engine)


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group2a` and build the model on these tables.
